# WP34 — Multi-Step Proof Tree Search (v0.4: The Autonomous Mathematician)
## ProofTree · TacticPrior · ProofTreeSearcher

Demonstrates **WP34**: best-first search over tactic sequences for Lean 4 theorem proving,
extending WP33's linear-retry approach to a branching tree search with an online-updated
tactic prior.

### What WP34 Introduces

| Component | Role |
|-----------|------|
| **TacticPrior** | Prior over tactic names; updated online from search outcomes |
| **ProofNode** | One node in the search tree (partial proof + tactic applied) |
| **ProofTree** | Best-first search tree; expands highest-priority open nodes |
| **ProofTreeSearcher** | Orchestrates full search; records `ProofSearchRecord` |

### Theoretical Grounding
> *"The first ultraintelligent machine is the last invention that man need ever make."*
> — I.J. Good (1965)

References: Newell & Simon (1972) GPS; Silver et al. (2016) AlphaGo; Lample & Charton (2019)

Runtime: **~2 min** (no GPU, no Lean required)


In [ ]:
# ── 0. Environment setup ──────────────────────────────────────────────────
import sys, os, importlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone -b wp16-notebook-only https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
import warnings; warnings.filterwarnings('ignore')
import time, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)
import prometheus
print(f'Prometheus version: {prometheus.__version__}')

In [ ]:
class MockLeanTool:
    '''Mock Lean 4 verifier: ring/simp/rfl tactics succeed, others fail.'''
    _using_mock = True
    PASSING = {'ring', 'simp', 'rfl', 'norm_num', 'decide', 'trivial'}
    def use(self, code):
        if 'sorry' in code:
            return "Proof contains 'sorry'"
        for t in self.PASSING:
            if f':= by\n  {t}' in code or f':= by {t}' in code:
                return None
        return 'unsolved goals'

lean = MockLeanTool()
print('LeanTool mode: sandboxed mock')

In [ ]:
# ── 1. TacticPrior demo ──────────────────────────────────────────────────
from prometheus.wp34_proof_tree_search import TacticPrior, _DEFAULT_LEAN4_PRIOR

prior = _DEFAULT_LEAN4_PRIOR
print('Default TacticPrior scores (top-8):')
top = sorted(prior.to_dict().items(), key=lambda x: -x[1])[:8]
for tac, sc in top:
    bar = chr(9608) * int(sc * 20)
    print(f'  {tac:<20} {sc:.2f}  {bar}')
print()
print('top_k([simp,ring,omega,cases], k=3):', prior.top_k(['simp','ring','omega','cases'], 3))


In [ ]:
# ── 2. ProofTree single-theorem demo ─────────────────────────────────────
from prometheus.wp34_proof_tree_search import ProofTree, _DEFAULT_LEAN4_PRIOR
import time

theorem  = 'theorem add_comm_nat (a b : Nat) : a + b = b + a'
tactics  = ['simp','ring','omega','rfl','norm_num','decide','trivial']

t0 = time.time()
tree = ProofTree(theorem + ' := by', lean, _DEFAULT_LEAN4_PRIOR, max_depth=4, max_nodes=80)
soln = tree.best_first_search(tactics, branching_factor=5)
elapsed = time.time() - t0
st = tree.statistics()
print('ProofTree results:')
print(f'  Solved:   {soln is not None}')
print(f'  Nodes:    {st["nodes_expanded"]}')
print(f'  Depth:    {st["solution_depth"]}')
print(f'  Tactics:  {st["solution_tactics"]}')
print(f'  Time:     {elapsed:.3f}s')


In [ ]:
# ── 3. Batch search ──────────────────────────────────────────────────────
from prometheus.wp34_proof_tree_search import ProofTreeSearcher

THEOREMS = [
    'theorem add_zero (n : Nat) : n + 0 = n',
    'theorem zero_add (n : Nat) : 0 + n = n',
    'theorem mul_one  (n : Nat) : n * 1 = n',
    'theorem add_comm_nat (a b : Nat) : a + b = b + a',
    'theorem mul_comm_nat (a b : Nat) : a * b = b * a',
    'theorem add_assoc_nat (a b c : Nat) : a + b + c = a + (b + c)',
]
searcher = ProofTreeSearcher(lean_tool=lean, max_nodes=80, branching_factor=5)
records  = searcher.batch_search(THEOREMS)
s = searcher.summary()
print(f'Batch: {s["n_proved"]}/{s["n_theorems"]} proved  success_rate={s["success_rate"]:.0%}  avg_nodes={s["avg_nodes"]:.1f}')
print()
for r in records:
    ok  = 'PASS' if r.success else 'FAIL'
    dep = str(r.solution_depth) if r.solution_depth else '-'
    print(f'  [{ok}] depth={dep}  nodes={r.nodes_expanded}  {r.theorem[:55]}')


In [ ]:
# ── 4. Visualisation ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Prior evolution
ax = axes[0]
SHOW = ['simp','ring','omega','rfl','norm_num','decide']
before = _DEFAULT_LEAN4_PRIOR.to_dict()
after  = searcher.tactic_prior.to_dict()
xr = range(len(SHOW))
ax.bar([i-0.2 for i in xr],[before.get(t,0.1) for t in SHOW],0.4,label='Before',color='#2196F3',alpha=0.8)
ax.bar([i+0.2 for i in xr],[after.get(t,0.1) for t in SHOW], 0.4,label='After', color='#FF9800',alpha=0.8)
ax.set_xticks(list(xr)); ax.set_xticklabels(SHOW,rotation=30)
ax.set_ylabel('Prior Score'); ax.set_ylim(0,1)
ax.set_title('TacticPrior Before vs After', fontweight='bold'); ax.legend()

# Nodes per theorem
ax2 = axes[1]
colors = ['#4CAF50' if r.success else '#F44336' for r in records]
ax2.bar(range(len(records)),[r.nodes_expanded for r in records],color=colors,edgecolor='black',alpha=0.85)
ax2.set_xticks(range(len(records))); ax2.set_xticklabels([f'T{i+1}' for i in range(len(records))])
ax2.set_ylabel('Nodes Expanded')
ax2.set_title('Nodes Expanded per Theorem', fontweight='bold')
ax2.legend(handles=[mpatches.Patch(color='#4CAF50',label='Proved'),mpatches.Patch(color='#F44336',label='Failed')])

# Depth distribution
ax3 = axes[2]
depths = [r.solution_depth for r in records if r.solution_depth is not None]
if depths:
    ax3.hist(depths,bins=range(0,max(depths)+2),color='#9C27B0',edgecolor='black',alpha=0.85)
ax3.set_xlabel('Solution Depth'); ax3.set_ylabel('Count')
ax3.set_title('Solution Depth Distribution', fontweight='bold')

fig.suptitle('WP34: Multi-Step Proof Tree Search', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('wp34_proof_tree_search.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved wp34_proof_tree_search.png')


In [ ]:
# ── 5. Exit-criteria verification ─────────────────────────────────────────
from prometheus.wp34_proof_tree_search import verify_wp34_exit_criteria

criteria = verify_wp34_exit_criteria(records, searcher.tactic_prior)
print('WP34 Exit Criteria Verification')
print('=' * 58)
for c, ok in criteria.items():
    print(f'  {"PASS" if ok else "FAIL"}  {c}')
print()
if all(criteria.values()):
    print('All WP34 exit criteria satisfied.')


---
## Conclusions

**WP34** adds branching proof tree search to the theorem-proving stack:
- `TacticPrior` guides which tactics to try first; updated online from outcomes
- `ProofTree` uses best-first search with depth + priority budget
- Solution tactic paths are logged in `ProofSearchRecord` for WP35

### Foundation for WP35 — Meta-Learning from Failure
WP35 conditions the tactic prior on *difficulty-specific* failure rates across theorems.

### References
- Newell & Simon (1972) *General Problem Solver*
- Silver et al. (2016) *AlphaGo* — prior-guided MCTS
- Good (1965) *Speculations concerning the first ultraintelligent machine*
